# Assignment 8: WordNet-based Word Sense Disambiguation

## Objective
Apply WordNet-based Word Sense Disambiguation (WSD) to improve meaning interpretation in ambiguous sentences using the Lesk algorithm.

## 1. Install and Import Libraries

In [ ]:
%pip install nltk
import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Download required data
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('stopwords', quiet=True)

print("Libraries imported successfully!")

## 2. Explore WordNet

Understanding WordNet synsets and word meanings.

In [ ]:
# Explore polysemous words (words with multiple meanings)
ambiguous_words = ['bank', 'plant', 'bat', 'match', 'mouse']

print("WordNet Synsets for Ambiguous Words:")
print("=" * 80)

for word in ambiguous_words:
    synsets = wn.synsets(word)
    print(f"\n'{word}' has {len(synsets)} different meanings:")
    for i, syn in enumerate(synsets[:5], 1):  # Show first 5
        print(f"  {i}. {syn.name()}: {syn.definition()}")
        print(f"     Example: {syn.examples()[0] if syn.examples() else 'N/A'}")

## 3. Ambiguous Sentences

Creating test sentences with ambiguous words.

In [ ]:
# Test sentences with ambiguous words
test_sentences = [
    ("I went to the bank to deposit money.", "bank"),
    ("We sat by the river bank and enjoyed the view.", "bank"),
    ("The plant produces oxygen through photosynthesis.", "plant"),
    ("They built a new manufacturing plant near the city.", "plant"),
    ("The cricket bat was made of willow wood.", "bat"),
    ("A bat flew into the cave at night.", "bat"),
    ("She used a match to light the candle.", "match"),
    ("Our team won the football match yesterday.", "match"),
    ("I need to buy a new computer mouse.", "mouse"),
    ("The cat chased the mouse across the room.", "mouse")
]

print("Test Sentences with Ambiguous Words:")
print("=" * 80)
for i, (sentence, target) in enumerate(test_sentences, 1):
    print(f"{i}. {sentence}")
    print(f"   Target word: '{target}'\n")

## 4. Lesk Algorithm for Word Sense Disambiguation

The Lesk algorithm finds the correct sense by comparing context overlap with definitions.

In [ ]:
def disambiguate_word(sentence, target_word):
    """
    Apply Lesk algorithm to find the correct sense of target_word in sentence
    """
    # Tokenize sentence
    tokens = word_tokenize(sentence.lower())
    
    # Apply Lesk algorithm
    best_sense = lesk(tokens, target_word)
    
    return best_sense

print("Word Sense Disambiguation using Lesk Algorithm:")
print("=" * 80)

results = []
for sentence, target_word in test_sentences:
    sense = disambiguate_word(sentence, target_word)
    
    if sense:
        results.append({
            'Sentence': sentence,
            'Word': target_word,
            'Synset': sense.name(),
            'Definition': sense.definition(),
            'POS': sense.pos()
        })
        
        print(f"\nSentence: {sentence}")
        print(f"Target: '{target_word}'")
        print(f"Selected Sense: {sense.name()}")
        print(f"Definition: {sense.definition()}")
        print(f"Part of Speech: {sense.pos()}")
    else:
        print(f"\nSentence: {sentence}")
        print(f"Target: '{target_word}'")
        print(f"No sense found!")
    print("-" * 80)

## 5. Manual Word Sense Disambiguation

Implementing custom WSD using context similarity.

In [ ]:
def manual_lesk(sentence, target_word):
    """
    Manual implementation of simplified Lesk algorithm
    """
    # Tokenize and get context
    context = set(word_tokenize(sentence.lower()))
    context.discard(target_word.lower())
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    context = context - stop_words
    
    # Get all synsets for target word
    synsets = wn.synsets(target_word)
    
    best_sense = None
    max_overlap = 0
    
    for synset in synsets:
        # Get definition and example words
        definition = set(word_tokenize(synset.definition().lower()))
        examples = set()
        for example in synset.examples():
            examples.update(word_tokenize(example.lower()))
        
        # Calculate overlap
        signature = definition.union(examples) - stop_words
        overlap = len(context.intersection(signature))
        
        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = synset
    
    return best_sense, max_overlap

print("Manual WSD Implementation:")
print("=" * 80)

for sentence, target_word in test_sentences[:5]:  # Test on first 5
    sense, overlap = manual_lesk(sentence, target_word)
    
    print(f"\nSentence: {sentence}")
    print(f"Word: '{target_word}'")
    if sense:
        print(f"Best Sense: {sense.name()}")
        print(f"Definition: {sense.definition()}")
        print(f"Context Overlap Score: {overlap}")
    else:
        print("No sense found")
    print("-" * 80)

## 6. WordNet Similarity Measures

Calculating semantic similarity between word senses.

In [ ]:
# Calculate similarity between word senses
word_pairs = [
    ('bank.n.01', 'bank.n.02'),  # financial vs river bank
    ('plant.n.01', 'plant.n.02'),  # organism vs factory
    ('bat.n.01', 'bat.n.05'),  # animal vs sports equipment
]

print("WordNet Semantic Similarity:")
print("=" * 80)

for syn1_name, syn2_name in word_pairs:
    syn1 = wn.synset(syn1_name)
    syn2 = wn.synset(syn2_name)
    
    print(f"\n{syn1.name()}: {syn1.definition()}")
    print(f"{syn2.name()}: {syn2.definition()}")
    
    # Path similarity
    path_sim = syn1.path_similarity(syn2)
    print(f"Path Similarity: {path_sim:.4f}" if path_sim else "Path Similarity: None")
    
    # Wu-Palmer similarity
    wup_sim = syn1.wup_similarity(syn2)
    print(f"Wu-Palmer Similarity: {wup_sim:.4f}" if wup_sim else "Wu-Palmer Similarity: None")
    print("-" * 80)

## 7. Evaluation and Summary

Summary of WSD results.

In [ ]:
# Create summary DataFrame
if results:
    df_results = pd.DataFrame(results)
    print("WSD Results Summary:")
    print("=" * 80)
    print(df_results.to_string(index=False))
    
    # Statistics
    print(f"\n{'=' * 80}")
    print(f"Total sentences processed: {len(test_sentences)}")
    print(f"Successfully disambiguated: {len(results)}")
    print(f"Success rate: {len(results)/len(test_sentences)*100:.1f}%")
    
    # Unique word senses
    unique_words = df_results['Word'].unique()
    print(f"Unique ambiguous words: {len(unique_words)}")
    print(f"Words: {', '.join(unique_words)}")

print("\n" + "=" * 80)
print("✓ Word Sense Disambiguation completed successfully!")